# 🌿 CropVision — Week 3: Transfer Learning & Hyperparameter Tuning
**Infotact DS/ML Internship | Project 2: Agriculture & Smart Farming**

---

### 📋 Week 3 Objectives
- Load **ResNet50** pretrained on ImageNet (1M+ images)
- **Freeze base layers** — keep ResNet50's learned features
- Attach a **custom classification head** for our 38 plant disease classes
- **Fine-tune** with a low learning rate to push accuracy above 90%
- Experiment with **hyperparameters** — learning rate, batch size, unfreezing layers
- Compare ResNet50 performance vs Week 2 custom CNN

### ⚠️ Commit Reminder
```
transfer: ResNet50 loaded, base layers frozen, custom head attached
transfer: phase 1 training complete, val_acc=0.XX
model-tuning: unfroze last 30 layers, fine-tuning with lr=1e-5
model-tuning: lr experiment — best lr found: X
eval: ResNet50 vs CustomCNN comparison complete
```

---
## 0. Imports & Config

In [ ]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import ResNet50, MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

plt.rcParams['figure.facecolor'] = '#f9f9f9'
plt.rcParams['axes.facecolor']   = '#ffffff'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)

print('✅ Libraries imported')
print(f'   TensorFlow : {tf.__version__}')
print(f'   GPU        : {len(tf.config.list_physical_devices("GPU")) > 0}')

---
## 1. Reload Datasets

⚠️ **Important for Transfer Learning:**  
ResNet50 was trained with **ImageNet normalization** (not simple [0,1] division).  
We must apply `preprocess_input` from Keras — it does the exact normalization ResNet50 expects.

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

DATA_DIR   = Path('data/PlantVillage')
IMG_SIZE   = [224, 224]
BATCH_SIZE = 32

with open('outputs/class_names.json') as f:
    class_names = json.load(f)

NUM_CLASSES  = len(class_names)
label_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_label = {v: k for k, v in label_to_idx.items()}


def stratified_split(data_dir, train_r=0.70, val_r=0.15, seed=42):
    random.seed(seed)
    splits = {'train': [], 'val': [], 'test': []}
    for class_name in sorted([d.name for d in data_dir.iterdir() if d.is_dir()]):
        class_dir = data_dir / class_name
        all_imgs  = (
            list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.JPG')) +
            list(class_dir.glob('*.png')) + list(class_dir.glob('*.PNG'))
        )
        random.shuffle(all_imgs)
        n       = len(all_imgs)
        n_train = int(n * train_r)
        n_val   = int(n * val_r)
        splits['train'].extend([(p, class_name) for p in all_imgs[:n_train]])
        splits['val'  ].extend([(p, class_name) for p in all_imgs[n_train:n_train + n_val]])
        splits['test' ].extend([(p, class_name) for p in all_imgs[n_train + n_val:]])
    return splits


# ── Use ResNet50's preprocess_input instead of /255 ──
def load_and_preprocess_resnet(image_path, label_idx):
    raw = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img = resnet_preprocess(img)          # ← ResNet50-specific normalization
    return img, tf.one_hot(label_idx, NUM_CLASSES)

def augment_tf(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    return img, label

def make_dataset(split_data, shuffle=False, augment=False):
    paths  = [str(p) for p, _ in split_data]
    labels = [label_to_idx[l] for _, l in split_data]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=RANDOM_SEED)
    ds = ds.map(load_and_preprocess_resnet, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_tf, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


splits   = stratified_split(DATA_DIR)
train_ds = make_dataset(splits['train'], shuffle=True,  augment=True)
val_ds   = make_dataset(splits['val'],   shuffle=False, augment=False)
test_ds  = make_dataset(splits['test'],  shuffle=False, augment=False)

print(f'✅ Datasets ready (with ResNet50 preprocessing):')
print(f'   train_ds : {len(train_ds)} batches')
print(f'   val_ds   : {len(val_ds)} batches')
print(f'   test_ds  : {len(test_ds)} batches')

---
## 2. What is Transfer Learning?

```
ResNet50 trained on ImageNet (1.2M images, 1000 classes)
           ↓
It already knows: edges, textures, shapes, object parts
           ↓
We FREEZE these base layers — keep all that knowledge
           ↓
We REPLACE the top (output) layer with our own:
   Dense(256) → Dropout → Dense(38 classes, softmax)
           ↓
We only TRAIN the new top layers (much faster!)
           ↓
Phase 2: UNFREEZE last N layers and fine-tune everything
         with a very small learning rate
```

**Why does this work?**  
Leaf disease detection needs the same low-level features (edges, textures, colors) that ResNet50 already learned from millions of photos. We just need to teach it to recognize plant-specific disease patterns on top of that.

---
## 3. Build Transfer Learning Model — Phase 1 (Frozen Base)

In [ ]:
def build_transfer_model(num_classes, base_model_name='resnet50',
                         input_shape=(224, 224, 3), dropout_rate=0.5):
    """
    Build a Transfer Learning model.

    Phase 1 strategy:
        - Load ResNet50 with ImageNet weights
        - Freeze ALL base layers (trainable=False)
        - Add custom head: GlobalAveragePooling → Dense → Dropout → Output
        - Train only the new head at lr=1e-3

    Args:
        num_classes     : number of output classes
        base_model_name : 'resnet50' or 'mobilenetv2'
        input_shape     : (H, W, C)
        dropout_rate    : dropout before output layer

    Returns:
        model, base_model (base_model needed for unfreezing in Phase 2)
    """
    # ── Load pretrained base ──
    if base_model_name == 'resnet50':
        base_model = ResNet50(
            weights     = 'imagenet',    # download pretrained weights
            include_top = False,          # remove ImageNet's 1000-class head
            input_shape = input_shape
        )
    else:
        base_model = MobileNetV2(
            weights     = 'imagenet',
            include_top = False,
            input_shape = input_shape
        )

    # ── Freeze all base layers ──
    base_model.trainable = False

    # ── Build model using Functional API ──
    inputs = keras.Input(shape=input_shape)
    x      = base_model(inputs, training=False)  # training=False = BatchNorm in inference mode

    # Custom classification head
    x = layers.GlobalAveragePooling2D()(x)        # (7,7,2048) → 2048
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name=f'CropVision_{base_model_name.upper()}')

    model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=1e-3),
        loss      = 'categorical_crossentropy',
        metrics   = ['accuracy']
    )

    return model, base_model


# Build ResNet50 transfer model
resnet_model, resnet_base = build_transfer_model(NUM_CLASSES, base_model_name='resnet50')

# Show trainable vs frozen
total_layers     = len(resnet_model.layers)
trainable_layers = len([l for l in resnet_model.layers if l.trainable])
frozen_layers    = total_layers - trainable_layers

print('✅ ResNet50 Transfer Learning model built')
print(f'   Total layers     : {total_layers}')
print(f'   Trainable layers : {trainable_layers}  ← only our custom head')
print(f'   Frozen layers    : {frozen_layers}   ← all of ResNet50 base')
print()
print(f'   Total params     : {resnet_model.count_params():,}')
trainable_p = sum([tf.size(w).numpy() for w in resnet_model.trainable_weights])
print(f'   Trainable params : {trainable_p:,}  ← only head params')

---
## 4. Phase 1 Training — Train Custom Head Only

Train only our new classification head on top of the frozen ResNet50.  
This is fast (few trainable params) and gets us a strong starting point.

In [ ]:
cb_ckpt_p1 = callbacks.ModelCheckpoint(
    'models/resnet50_phase1.keras',
    monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
)
cb_early_p1 = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5,
    restore_best_weights=True, verbose=1
)
cb_csv_p1 = callbacks.CSVLogger('outputs/training_log_resnet_phase1.csv')

print('🚀 Phase 1: Training custom head (base frozen)...')
print(f'   Trainable params: {sum([tf.size(w).numpy() for w in resnet_model.trainable_weights]):,}')
print()

history_p1 = resnet_model.fit(
    train_ds,
    epochs          = 15,
    validation_data = val_ds,
    callbacks       = [cb_ckpt_p1, cb_early_p1, cb_csv_p1],
    verbose         = 1
)

best_p1_acc = max(history_p1.history['val_accuracy'])
print(f'\n✅ Phase 1 complete!')
print(f'   Best val_accuracy: {best_p1_acc:.4f}')

---
## 5. Phase 2 — Unfreeze & Fine-Tune

Now we **unfreeze the last N layers** of ResNet50 and train them together with our head at a very low learning rate.

**Why a very low learning rate (1e-5)?**  
The ResNet50 base already has good weights from ImageNet. If we use a high learning rate, we'll destroy those weights with random gradients.  
A tiny learning rate makes small, careful adjustments — adapting ResNet50's knowledge to plant diseases.

In [ ]:
# ── Unfreeze last 30 layers of ResNet50 ──
UNFREEZE_LAYERS = 30   # experiment: try 20, 30, 50

# First unfreeze all
resnet_base.trainable = True

# Re-freeze everything except the last N layers
for layer in resnet_base.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

# Keep BatchNorm layers frozen (important!)
# Unfreezing BN in a pretrained model can destabilize training
for layer in resnet_base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Recompile with VERY low learning rate
resnet_model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-5),  # 100x smaller than Phase 1
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

# Count new trainable params
trainable_p2 = sum([tf.size(w).numpy() for w in resnet_model.trainable_weights])
trainable_l2 = len([l for l in resnet_base.layers if l.trainable])

print(f'✅ Phase 2 setup complete:')
print(f'   Unfroze last {UNFREEZE_LAYERS} layers of ResNet50')
print(f'   Trainable layers in base: {trainable_l2}')
print(f'   Total trainable params  : {trainable_p2:,}')
print(f'   Learning rate           : 1e-5  (very small — careful fine-tuning)')

In [ ]:
cb_ckpt_p2 = callbacks.ModelCheckpoint(
    'models/resnet50_phase2_best.keras',
    monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
)
cb_early_p2 = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7,
    restore_best_weights=True, verbose=1
)
cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3,
    min_lr=1e-7, verbose=1
)
cb_csv_p2 = callbacks.CSVLogger('outputs/training_log_resnet_phase2.csv')

print('🚀 Phase 2: Fine-tuning unfrozen layers + head...')
print()

history_p2 = resnet_model.fit(
    train_ds,
    epochs          = 30,
    validation_data = val_ds,
    callbacks       = [cb_ckpt_p2, cb_early_p2, cb_reduce_lr, cb_csv_p2],
    verbose         = 1
)

best_p2_acc = max(history_p2.history['val_accuracy'])
print(f'\n✅ Phase 2 complete!')
print(f'   Phase 1 best val_accuracy: {best_p1_acc:.4f}')
print(f'   Phase 2 best val_accuracy: {best_p2_acc:.4f}')
print(f'   Improvement from fine-tuning: +{best_p2_acc - best_p1_acc:.4f}')

---
## 6. Plot Phase 1 + Phase 2 Training Curves

In [ ]:
# Combine Phase 1 and Phase 2 histories
acc_p1     = history_p1.history['accuracy']
val_acc_p1 = history_p1.history['val_accuracy']
acc_p2     = history_p2.history['accuracy']
val_acc_p2 = history_p2.history['val_accuracy']

loss_p1     = history_p1.history['loss']
val_loss_p1 = history_p1.history['val_loss']
loss_p2     = history_p2.history['loss']
val_loss_p2 = history_p2.history['val_loss']

# Concatenate
all_acc     = acc_p1     + acc_p2
all_val_acc = val_acc_p1 + val_acc_p2
all_loss    = loss_p1    + loss_p2
all_val_loss= val_loss_p1+ val_loss_p2
epochs      = range(1, len(all_acc) + 1)
p2_start    = len(acc_p1) + 1   # where Phase 2 begins

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ResNet50 Transfer Learning — Phase 1 + Phase 2 Training Curves',
             fontsize=13, fontweight='bold')

for ax, train_vals, val_vals, ylabel, title in zip(
    axes,
    [all_acc, all_loss],
    [all_val_acc, all_val_loss],
    ['Accuracy', 'Loss'],
    ['Accuracy', 'Loss']
):
    ax.plot(epochs, train_vals, color='#27ae60', linewidth=2, label='Train')
    ax.plot(epochs, val_vals,   color='#2980b9', linewidth=2, linestyle='--', label='Val')

    # Mark phase boundary
    ax.axvline(p2_start, color='#e74c3c', linestyle=':', linewidth=2,
               label=f'Phase 2 starts (epoch {p2_start})')
    ax.fill_betweenx([min(train_vals + val_vals), max(train_vals + val_vals)],
                     1, p2_start, alpha=0.05, color='#27ae60', label='Phase 1')
    ax.fill_betweenx([min(train_vals + val_vals), max(train_vals + val_vals)],
                     p2_start, len(epochs), alpha=0.05, color='#e74c3c', label='Phase 2')

    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('outputs/training_curves_resnet.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/training_curves_resnet.png')
print(f'\n🎯 Final val_accuracy after fine-tuning: {best_p2_acc:.4f} ({best_p2_acc*100:.2f}%)')
if best_p2_acc >= 0.90:
    print('   ✅ TARGET ACHIEVED: >90% accuracy!')
else:
    print(f'   ⚡ Not yet at 90%. Gap: {0.90 - best_p2_acc:.4f} — try tuning in Section 7')

### 📝 Observation — Training Curves

> **Write your findings here:**
> - Did Phase 2 fine-tuning improve accuracy over Phase 1?
> - Was there any instability at the start of Phase 2 (common when unfreezing)?
> - Did the model reach >90% accuracy?
> - How does this compare to the Week 2 custom CNN?

*Example: "Phase 1 reached 0.87 val_acc. Fine-tuning in Phase 2 pushed it to 0.93 — exceeding the 90% target. A brief accuracy dip at epoch 16 (Phase 2 start) stabilized within 2 epochs."*

---
## 7. Hyperparameter Experiments

Systematically test different settings to find the best combination.  
We test **learning rates** — the most impactful hyperparameter.

In [ ]:
# ── Learning Rate Experiment ──
# Train 3 mini-versions with different LRs for 5 epochs each
# This helps us pick the best LR for the final full run

LR_CANDIDATES = [1e-4, 5e-5, 1e-5]
lr_results    = {}

print('🔬 LEARNING RATE EXPERIMENT')
print('=' * 45)
print('Training 3 models for 5 epochs each...\n')

for lr in LR_CANDIDATES:
    print(f'\n→ Testing lr = {lr}')

    # Build fresh model
    m, base = build_transfer_model(NUM_CLASSES)
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    m.compile(
        optimizer = keras.optimizers.Adam(learning_rate=lr),
        loss      = 'categorical_crossentropy',
        metrics   = ['accuracy']
    )

    hist = m.fit(
        train_ds,
        epochs          = 5,
        validation_data = val_ds,
        verbose         = 0
    )

    best_val = max(hist.history['val_accuracy'])
    lr_results[lr] = {'val_acc': best_val, 'history': hist}
    print(f'   Best val_accuracy (5 epochs): {best_val:.4f}')


# Plot LR comparison
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#27ae60', '#2980b9', '#e74c3c']

for (lr, result), color in zip(lr_results.items(), colors):
    epochs = range(1, 6)
    ax.plot(epochs, result['history'].history['val_accuracy'],
            color=color, linewidth=2, marker='o', label=f'lr={lr}')

ax.set_title('Learning Rate Comparison — Validation Accuracy (5 epochs)', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Accuracy')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('outputs/lr_experiment.png', dpi=150, bbox_inches='tight')
plt.show()

best_lr = max(lr_results, key=lambda k: lr_results[k]['val_acc'])
print(f'\n🏆 Best learning rate: {best_lr}')
print(f'   Val accuracy      : {lr_results[best_lr]["val_acc"]:.4f}')
print(f'\n💡 Use this lr in your final full training run if needed')

---
## 8. MobileNetV2 — Lightweight Alternative

**Why try MobileNetV2?**  
- Much smaller than ResNet50 (3.4M vs 25M parameters)
- Faster to train and deploy on mobile (perfect for our farmer app use-case)
- Often achieves comparable accuracy for image classification

We'll quickly train it and compare against ResNet50.

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

# MobileNetV2 needs its own preprocessing
def load_and_preprocess_mobilenet(image_path, label_idx):
    raw = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img = mobilenet_preprocess(img)       # scales to [-1, 1]
    return img, tf.one_hot(label_idx, NUM_CLASSES)

def make_dataset_mobilenet(split_data, shuffle=False, augment=False):
    paths  = [str(p) for p, _ in split_data]
    labels = [label_to_idx[l] for _, l in split_data]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=RANDOM_SEED)
    ds = ds.map(load_and_preprocess_mobilenet, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_tf, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds_mob = make_dataset_mobilenet(splits['train'], shuffle=True,  augment=True)
val_ds_mob   = make_dataset_mobilenet(splits['val'],   shuffle=False, augment=False)

# Build MobileNetV2
mobilenet_model, mobilenet_base = build_transfer_model(
    NUM_CLASSES, base_model_name='mobilenetv2'
)

cb_mob = callbacks.ModelCheckpoint(
    'models/mobilenetv2_best.keras',
    monitor='val_accuracy', save_best_only=True, mode='max', verbose=0
)
cb_early_mob = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5,
    restore_best_weights=True, verbose=1
)

print('🚀 Training MobileNetV2 (Phase 1 only for comparison)...')
print(f'   Params: {mobilenet_model.count_params():,}  (vs ResNet50: {resnet_model.count_params():,})')

history_mob = mobilenet_model.fit(
    train_ds_mob,
    epochs          = 15,
    validation_data = val_ds_mob,
    callbacks       = [cb_mob, cb_early_mob],
    verbose         = 1
)

best_mob_acc = max(history_mob.history['val_accuracy'])
print(f'\n✅ MobileNetV2 best val_accuracy: {best_mob_acc:.4f}')

---
## 9. Final Model Comparison — Custom CNN vs ResNet50 vs MobileNetV2

In [ ]:
# Load Week 2 best val_accuracy from saved CSV
import pandas as pd

try:
    cnn_log  = pd.read_csv('outputs/training_log_tuned_cnn.csv')
    cnn_best = cnn_log['val_accuracy'].max()
except:
    cnn_best = 0.75  # placeholder if file not found

# Summary table
models_summary = {
    'Custom CNN\n(Week 2)':       cnn_best,
    'ResNet50\nPhase 1':          best_p1_acc,
    'ResNet50\nPhase 2 (tuned)':  best_p2_acc,
    'MobileNetV2\nPhase 1':       best_mob_acc
}

fig, ax = plt.subplots(figsize=(10, 5))
colors  = ['#95a5a6', '#f39c12', '#27ae60', '#3498db']
bars    = ax.bar(models_summary.keys(), models_summary.values(),
                 color=colors, edgecolor='none', width=0.5)

# Add value labels on bars
for bar, val in zip(bars, models_summary.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.axhline(0.90, color='#e74c3c', linestyle='--', linewidth=1.5, label='90% target')
ax.set_ylim([0, 1.05])
ax.set_ylabel('Validation Accuracy', fontsize=12)
ax.set_title('Model Comparison — Validation Accuracy', fontsize=13, pad=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/model_comparison.png')

best_model_name = max(models_summary, key=models_summary.get)
best_val        = max(models_summary.values())
print(f'\n🏆 BEST MODEL: {best_model_name.replace(chr(10), " ")}')
print(f'   Validation Accuracy: {best_val:.4f}')
print(f'   → This model will be used in Week 4 for final evaluation')

### 📝 Observation — Model Comparison

> **Write your findings here:**
> - Which model performed best?
> - How much did Transfer Learning improve over the custom CNN?
> - Did fine-tuning (Phase 2) help significantly over Phase 1?
> - Would you recommend ResNet50 or MobileNetV2 for this use case and why?

*Example: "ResNet50 Phase 2 achieved 0.94 vs custom CNN's 0.76 — an 18% improvement. Fine-tuning added 4% over Phase 1. For a mobile farmer app, MobileNetV2 (0.91) is preferable — 90%+ accuracy with 7x fewer parameters."*

---
## 10. Save Best Model for Week 4

In [ ]:
# Load and save the best model
best_model_path = 'models/resnet50_phase2_best.keras'  # update if MobileNet won
best_model      = keras.models.load_model(best_model_path)

# Save in both formats for flexibility
best_model.save('models/cropvision_final_model.keras')     # recommended Keras format
best_model.save('models/cropvision_final_model.h5')        # legacy HDF5 format

print('💾 Final model saved:')
print('   models/cropvision_final_model.keras')
print('   models/cropvision_final_model.h5')
print()
print('⚠️  Remember: .h5 and .keras files are gitignored!')
print('   Only commit the notebook and outputs/ folder')
print()

# Save best model name for reference in Week 4
meta = {
    'best_model_name'   : 'ResNet50 Phase 2',
    'best_val_accuracy' : best_p2_acc,
    'num_classes'       : NUM_CLASSES,
    'input_shape'       : [224, 224, 3],
    'preprocessing'     : 'resnet50_preprocess_input'
}
with open('outputs/best_model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('💾 Saved: outputs/best_model_meta.json')

---
## 11. Week 3 Summary & Commit Checklist

### ✅ What we accomplished this week

| Task | Status |
|------|--------|
| ResNet50 loaded with ImageNet weights | ✅ |
| Phase 1 — custom head training | ✅ |
| Phase 2 — fine-tuning unfrozen layers | ✅ |
| Learning rate experiment (3 values) | ✅ |
| MobileNetV2 comparison | ✅ |
| Model comparison chart | ✅ |
| Best model saved for Week 4 | ✅ |

### 📁 Outputs generated
```
outputs/
    training_curves_resnet.png
    lr_experiment.png
    model_comparison.png
    training_log_resnet_phase1.csv
    training_log_resnet_phase2.csv
    best_model_meta.json
models/
    cropvision_final_model.keras  (gitignored)
    cropvision_final_model.h5     (gitignored)
```

### 🔀 Commit now!
```bash
# Kernel → Restart & Clear Output first!
git add Week3_CropVision_TransferLearning.ipynb outputs/
git commit -m 'transfer: ResNet50 fine-tuned, val_acc=0.XX — exceeds 90% target'
git push origin main
```